# Reconcile latest FDA indications with existing MOAlmanac records

This notebook prototypes the drug-level identity-mapping step: classify indications as `matched`, `new`, `not_found`, or `uncertain` before downstream new-curation and revision assessment workflows.

In [1]:
import json, os
from collections import Counter
from pathlib import Path
from pprint import pprint
from dotenv import find_dotenv, load_dotenv
from moalmanac_fda_curation.core.extract_indications_from_fda_label import (
    DEFAULT_MAX_TOKENS, DEFAULT_MODEL, extract_indications_from_label_url,
)
from moalmanac_fda_curation.core.reconcile_indications import (
    build_reconciliation_prompt, indexed_latest_indications,
    load_existing_indications, reconcile_indications,
)
from moalmanac_fda_curation.core.revise_indication import (
    assess_update, build_assessment_prompt, events_since,
)

## Load environment and configure inputs

This first evaluation uses Opdivo. The existing records come from MOAlmanac, while the latest label URL is selected deterministically from the newest event in the local Opdivo changelog. Refresh that changelog first if it may be stale.

In [2]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")
PROJECT_ROOT = Path(env_path).parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
EXISTING_INDICATIONS_JSON = WORKSPACE_ROOT / "moalmanac-db/referenced/indications.json"
CHANGELOG_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs/Opdivo-bla125554-section1-changelog.json"
EXTRACTION_DIR = PROJECT_ROOT / "analyses/reconciliation-inputs/opdivo-current-label"

changelog = json.loads(CHANGELOG_JSON.read_text())
latest_event = max(changelog["events"], key=lambda event: (event["date"], event["event_number"]))
LATEST_LABEL_URL = latest_event["label_url"]
print("Latest changelog date:", latest_event["date"])
print("Latest label URL:", LATEST_LABEL_URL)

Latest changelog date: 2026-03-20
Latest label URL: http://www.accessdata.fda.gov/drugsatfda_docs/label/2026/125554s135lbl.pdf


## Regenerate latest-label indications

This is the costly step: it downloads the newest label and calls Anthropic using the repository's current extraction prompt. Run it whenever the label or extraction logic changes. Keep `REGENERATE = False` when you only want to rerun the mapping experiment against the saved extraction.

In [3]:
REGENERATE = False

if REGENERATE:
    extraction_paths = extract_indications_from_label_url(
        label_url=LATEST_LABEL_URL,
        brand_name="Opdivo",
        generic_name="nivolumab",
        application_number="BLA125554",
        labels_dir=EXTRACTION_DIR / "labels",
        indications_dir=EXTRACTION_DIR / "intermediate",
        model=DEFAULT_MODEL,
        max_tokens=DEFAULT_MAX_TOKENS,
        overwrite=True,
    )
else:
    extraction_paths = {
        "claude_chunked_indication_fields": EXTRACTION_DIR / "intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json"
    }

LATEST_EXTRACTION_JSON = extraction_paths["claude_chunked_indication_fields"]
print("Extraction artifact:", LATEST_EXTRACTION_JSON)

Extraction artifact: /Users/sabrina/Documents/VA/MOAlmanac/moalmanac-fda-curation/analyses/reconciliation-inputs/opdivo-current-label/intermediate/Opdivo-BLA125554-claude_chunked_indication_fields.json


## Load existing and freshly extracted indications

`biomarker_only=True` approximates the current MOAlmanac FDA-indication scope. Original extraction indexes are preserved so the mapping output can be traced back to this artifact.

In [4]:
existing_indications = load_existing_indications(EXISTING_INDICATIONS_JSON, "doc:fda.opdivo")
latest_payload = json.loads(LATEST_EXTRACTION_JSON.read_text())
latest_indications = indexed_latest_indications(latest_payload, biomarker_only=True)
print(f"Existing MOAlmanac indications: {len(existing_indications)}")
print(f"Latest in-scope label indications: {len(latest_indications)}")

Existing MOAlmanac indications: 5
Latest in-scope label indications: 8


## Inspect both indication sets

In [5]:
print("EXISTING")
for item in existing_indications:
    print(f"\n{item['id']}: {item['indication']}")
print("\nLATEST")
for item in latest_indications:
    print(f"\n[{item['latest_indication_index']}]: {item['indication']}")

EXISTING

ind:fda.opdivo:0: OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.

ind:fda.opdivo:1: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.

ind:fda.opdivo:2: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic or recurrent non-small cell lung cancer with no EGFR or ALK genomic tumor aberrations as first-line t

## Inspect the exact reconciliation prompt

In [6]:
prompt = build_reconciliation_prompt(existing_indications, latest_indications)
print(prompt)

# Task

Reconcile the existing MOAlmanac indications for one drug with indications
extracted from its latest FDA label.

# Existing MOAlmanac indications

```json
[
  {
    "id": "ind:fda.opdivo:0",
    "indication": "OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery."
  },
  {
    "id": "ind:fda.opdivo:1",
    "indication": "OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab."
  },
  {
    "id

## Run reconciliation

Python verifies referenced IDs and indexes, classification shape, uniqueness, and complete coverage. Clinical correctness remains a curator judgment.

In [7]:
reconciliation = reconcile_indications(existing_indications, latest_indications)
print("Verified:", reconciliation["verified"])
print("Verification errors:", reconciliation["verification_errors"])
print("Classification counts:", Counter(item["classification"] for item in reconciliation["mappings"]))

Verified: True
Verification errors: []
Classification counts: Counter({'matched': 5, 'new': 3})


## Review mappings

In [8]:
for mapping in reconciliation["mappings"]:
    print("\n" + "=" * 80)
    print("Classification:", mapping["classification"])
    print("Existing:", mapping["existing_indication_id"])
    print("Latest index:", mapping["latest_indication_index"])
    print("Reason:", mapping["reason"])
    if mapping["existing_indication"]:
        print("Existing text:", mapping["existing_indication"]["indication"])
    if mapping["latest_indication"]:
        print("Latest text:", mapping["latest_indication"]["indication"])


Classification: matched
Existing: ind:fda.opdivo:0
Latest index: 4
Reason: Both describe neoadjuvant treatment with platinum-doublet chemotherapy for resectable NSCLC (tumors >=4 cm or node positive) with no EGFR/ALK alterations, followed by adjuvant single-agent OPDIVO after surgery.
Existing text: OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.
Latest text: Opdivo is a programmed death receptor-1 (PD-1)-blocking antibody indicated, in combination with platinum-doublet chemotherapy, for the neoadjuvant treatment of adult patients with resectable (tumors ≥4 cm or node positive) NSCLC and no known epidermal growth factor receptor (EGFR) mutations or anapl

## Prepare revision assessments for matched indications

Reconciliation has established identity. Each `matched` existing MOAlmanac indication is now assessed against changelog events after the document's last curation date. The mapped latest string remains available for curator comparison but is not sent to the revision LLM.

In [9]:
MOA_DOCUMENT_JSON = PROJECT_ROOT / "analyses/revisions/opdivo-current-record/document.json"
moa_document = json.loads(MOA_DOCUMENT_JSON.read_text())
LAST_CURATION_DATE = moa_document["publication_date"]
candidate_events = events_since(changelog, LAST_CURATION_DATE)
matched_mappings = [
    mapping for mapping in reconciliation["mappings"]
    if mapping["classification"] == "matched"
]
print("Last curation date:", LAST_CURATION_DATE)
print("Post-curation changelog events:", len(candidate_events))
print("Matched indications to assess:", len(matched_mappings))

Last curation date: 2025-04-11
Post-curation changelog events: 27
Matched indications to assess: 5


## Inspect one exact revision-assessment prompt

Change `ASSESSMENT_PREVIEW_ID` to inspect another matched indication. The prompt contains the original indication object and post-curation changelog events only.

In [10]:
ASSESSMENT_PREVIEW_ID = matched_mappings[0]["existing_indication_id"]
preview_mapping = next(
    mapping for mapping in matched_mappings
    if mapping["existing_indication_id"] == ASSESSMENT_PREVIEW_ID
)
print(build_assessment_prompt(preview_mapping["existing_indication"], candidate_events))

# Task

Decide whether later versions of the label's Indications and Usage section
clinically changed this existing MOAlmanac FDA indication.

# Target indication

```json
{
  "id": "ind:fda.opdivo:0",
  "document_id": "doc:fda.opdivo",
  "indication": "OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.",
  "initial_approval_date": "2024-10-03",
  "initial_approval_url": "https://www.accessdata.fda.gov/drugsatfda_docs/label/2024/125554s127lbl.pdf",
  "description": "The U.S. Food and Drug Administration granted approval to nivolumab in combination with platinum-doublet chemotherapy for the neoadjuvant treatment, followed by single-agent nivolumab as adjuvant

## Assess every matched indication for revision

This makes one Anthropic call per matched indication. It runs only assessment and evidence selection; it does not propose or write revised JSON. Opdivo can require lengthy verbatim evidence, so the assessment output budget is set explicitly below. A failure for one indication is recorded without preventing assessment of the remaining matches.

In [11]:
RUN_REVISION_ASSESSMENTS = True
ASSESSMENT_MAX_TOKENS = 8000
revision_assessments = {}
revision_assessment_errors = {}
if RUN_REVISION_ASSESSMENTS:
    for mapping in matched_mappings:
        indication_id = mapping["existing_indication_id"]
        print("Assessing", indication_id)
        try:
            revision_assessments[indication_id] = {
                "latest_indication_index": mapping["latest_indication_index"],
                "latest_indication": mapping["latest_indication"],
                **assess_update(
                    mapping["existing_indication"],
                    candidate_events,
                    max_tokens=ASSESSMENT_MAX_TOKENS,
                ),
            }
        except Exception as error:
            revision_assessment_errors[indication_id] = error
            print("  Failed:", type(error).__name__, error)
print("Completed assessments:", len(revision_assessments))
print("Failed assessments:", len(revision_assessment_errors))

Assessing ind:fda.opdivo:0
Assessing ind:fda.opdivo:1
Assessing ind:fda.opdivo:2
Assessing ind:fda.opdivo:3
Assessing ind:fda.opdivo:4
Completed assessments: 5
Failed assessments: 0


## Review revision assessments and verified spans

In [12]:
for indication_id, result in revision_assessments.items():
    assessment = result["assessment"]
    print("\n" + "=" * 80)
    print("Existing indication:", indication_id)
    print("Mapped latest index:", result["latest_indication_index"])
    print("Status:", assessment["status"])
    print("Verified:", result["verified"])
    print("What changed:", assessment["what_changed"])
    print("Reasoning:", assessment["reasoning"])
    print("Uncertainties:", assessment["uncertainties"])
    if result.get("verification_errors"):
        print("Verification errors:", result["verification_errors"])
    print("Verified target spans:")
    for span in result["scoped_evidence"]:
        pprint(span)


Existing indication: ind:fda.opdivo:0
Mapped latest index: 4
Status: updated
Verified: True
What changed: ["The required test for EGFR/ALK genomic tumor aberrations changed from 'FDA-approved test' to 'FDA-authorized test'"]
Reasoning: Event 234 directly modifies language that applies to the target indication. The target indication specifies 'no known EGFR mutations or ALK rearrangements' as a biomarker requirement for neoadjuvant NSCLC treatment. Event 234 changes the test designation from 'FDA-approved test' to 'FDA-authorized test' in the context of NSCLC indications with EGFR/ALK requirements. This represents a clinically meaningful change to the required testing framework for patient selection. All other events are either formatting changes (removal of bullet points, line breaks) or concern separate indications (esophageal cancer PD-L1 requirements, Hodgkin lymphoma combination therapy, etc.) that do not modify the target neoadjuvant NSCLC indication.
Uncertainties: []
Verified t

## Optional: save the experimental reconciliation

In [ ]:
# output = PROJECT_ROOT / "analyses/revisions/opdivo-reconciliation.json"
# output.parent.mkdir(parents=True, exist_ok=True)
# output.write_text(json.dumps(reconciliation, indent=2) + "\n")
# output